In [ ]:
# Run this cell first after kernel start (defines client and model).
from pathlib import Path
import os

from dotenv import load_dotenv
from anthropic import Anthropic


def _project_root() -> Path:
    here = Path.cwd().resolve()
    for d in [here, *here.parents[:15]]:
        if (d / "tools.ipynb").is_file():
            return d
    return here


_PROJECT = _project_root()
load_dotenv(_PROJECT / ".env")
load_dotenv()

if not os.environ.get("ANTHROPIC_API_KEY"):
    raise RuntimeError(
        "ANTHROPIC_API_KEY is not set. Put it in a .env file in this folder, or export it before "
        "starting Jupyter / Cursor, then restart the kernel."
    )

client = Anthropic()
model = "claude-haiku-4-5"

In [ ]:
# Run second (defines get_current_datetime and get_current_datetime_schema).
from datetime import datetime


def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)


get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S",
            }
        },
        "required": [],
    },
}

In [ ]:
# Run third. Requires the two cells above in this session.
import time

_need = ("client", "model", "get_current_datetime_schema", "get_current_datetime")
_missing = [n for n in _need if n not in globals()]
if _missing:
    raise RuntimeError(
        f"Missing: {', '.join(_missing)}. Run the first cell (client), then the second (tool), then this one."
    )

messages = [
    {
        "role": "user",
        "content": "What is the exact time, formatted as HH:MM:SS?",
    }
]

t0 = time.perf_counter()
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)
print(f"Initial API call took {time.perf_counter() - t0:.2f}s; stop_reason={response.stop_reason}")

iteration = 0
while response.stop_reason == "tool_use":
    iteration += 1
    iter_start = time.perf_counter()
    messages.append({"role": "assistant", "content": response.content})
    tool_results = []
    for block in response.content:
        if block.type != "tool_use":
            continue
        if block.name == "get_current_datetime":
            args = block.input if isinstance(block.input, dict) else {}
            fmt = args.get("date_format") or "%H:%M:%S"
            result = get_current_datetime(date_format=fmt)
            tool_results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": result,
                }
            )
    messages.append({"role": "user", "content": tool_results})
    api_start = time.perf_counter()
    response = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
        tools=[get_current_datetime_schema],
    )
    print(
        f"Tool loop {iteration}: model round-trip {time.perf_counter() - api_start:.2f}s, "
        f"total loop {time.perf_counter() - iter_start:.2f}s, stop_reason={response.stop_reason}"
    )

final_text_start = time.perf_counter()
for block in response.content:
    if getattr(block, "type", None) == "text" and getattr(block, "text", None):
        print(block.text)
print(f"Final response parse took {time.perf_counter() - final_text_start:.3f}s")
